# Practice Lab: Explainable AI (Lesson 15.2)

Module 15 . 3 exercises . LIME + SHAP force plots + XAI dashboard


# Lesson 15.2: Explainable AI (XAI)

3 exercises: 1E + 1M + 1C

Module 15: Ethics, Safety, Governance & Explainability


In [ ]:
import numpy as np
from collections import Counter, defaultdict


---
## Exercise 1 (Easy): LIME on 10 Texts


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
from collections import Counter

class SC:
    POS={"excellent":0.40,"great":0.30,"good":0.20,"love":0.35,"amazing":0.38,
        "fantastic":0.36,"helpful":0.25,"clear":0.22,"best":0.32,"recommend":0.28,
        "enjoyed":0.26,"perfect":0.35}
    NEG={"terrible":-0.40,"bad":-0.30,"boring":-0.28,"hate":-0.35,"waste":-0.32,
        "confusing":-0.25,"slow":-0.18,"poor":-0.30,"awful":-0.38,"disappointing":-0.28}
    def predict(self,t):
        s=0.5
        for w in t.lower().split(): s+=self.POS.get(w.strip(".,!?;:"),0)+self.NEG.get(w.strip(".,!?;:"),0)
        return max(0,min(1,s))

class LIME:
    def __init__(self,c): self.c=c
    def explain(self,t):
        ws=t.split(); base=self.c.predict(t); imp={}
        for i,w in enumerate(ws):
            p=" ".join(x for j,x in enumerate(ws) if j!=i)
            c=w.strip(".,!?;:"); imp[c]=round(base-self.c.predict(p),4)
        return sorted(imp.items(),key=lambda x:-abs(x[1]))

clf=SC(); lime=LIME(clf)
reviews=["The course was excellent and the projects were great",
    "I love how clear the explanations are in this course",
    "Terrible experience the content was boring and slow",
    "Good course but some sections were confusing",
    "This is the best GenAI course I have ever taken amazing",
    "The instructor was helpful and the examples were perfect",
    "Bad course waste of time and money very disappointing",
    "I enjoyed the hands-on labs they were fantastic",
    "The course was difficult but I learned a lot good content",
    "Poor quality videos and confusing structure awful experience"]

print("LIME on 10 Reviews:")
atw=[]
for i,r in enumerate(reviews):
    score=clf.predict(r); label="POS" if score>=0.5 else "NEG"
    top3=lime.explain(r)[:3]; atw.extend([w for w,_ in top3])
    print(f"  {i+1}. [{label} {score:.2f}] {r[:50]}...")
    for w,imp in top3: print(f"     {w:<13} {'+'if imp>0 else ''}{imp:.4f}")
print(f"\nTop driver words:")
for w,c in Counter(atw).most_common(8): print(f"  {w:<13} {c}x")


</details>


---
## Exercise 2 (Medium): SHAP + Force Plot


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
from collections import defaultdict

class SHAP:
    POS={"excellent":0.40,"great":0.30,"good":0.20,"love":0.35,"amazing":0.38,
        "fantastic":0.36,"helpful":0.25,"clear":0.22,"best":0.32}
    NEG={"terrible":-0.40,"bad":-0.30,"boring":-0.28,"hate":-0.35,"waste":-0.32,
        "confusing":-0.25,"slow":-0.18,"poor":-0.30,"awful":-0.38,"disappointing":-0.28}
    BASE=0.5
    def compute(self,text):
        ws=text.split(); sv={}
        for w in ws:
            c=w.strip(".,!?;:").lower(); v=self.POS.get(c,0)+self.NEG.get(c,0)
            if v!=0: sv[c]=round(v,4)
        pred=max(0,min(1,self.BASE+sum(sv.values())))
        return {"sv":sv,"pred":round(pred,4),"err":round(abs(self.BASE+sum(sv.values())-pred),6)}

s=SHAP()
reviews=["The course was excellent and the projects were great",
    "Terrible experience boring and slow",
    "I love how clear and helpful the examples are",
    "Bad course disappointing waste of time",
    "This is the best amazing GenAI course fantastic"]

print("SHAP Force Plots:")
gs=defaultdict(list)
for i,r in enumerate(reviews):
    res=s.compute(r); label="POS" if res["pred"]>=0.5 else "NEG"
    print(f"\n  {i+1}. pred={res['pred']} ({label}) err={res['err']}")
    for w,v in sorted(res["sv"].items(),key=lambda x:-abs(x[1])):
        gs[w].append(abs(v))
        print(f"     {w:<13} {v:>+.4f}")

print("\n  Global |SHAP|:")
gi={w:round(np.mean(v),4) for w,v in gs.items()}
for w,v in sorted(gi.items(),key=lambda x:-x[1])[:8]:
    print(f"    {w:<13} {v:.4f}")


</details>


---
## Exercise 3 (Challenge): XAI Dashboard


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np

class XAI:
    POS={"excellent":0.40,"great":0.30,"good":0.20,"love":0.35,"amazing":0.38,
        "fantastic":0.36,"helpful":0.25,"clear":0.22,"best":0.32}
    NEG={"terrible":-0.40,"bad":-0.30,"boring":-0.28,"hate":-0.35,"waste":-0.32,
        "confusing":-0.25,"poor":-0.30,"awful":-0.38,"disappointing":-0.28}
    def predict(self,t):
        s=0.5
        for w in t.lower().split():
            c=w.strip(".,!?;:"); s+=self.POS.get(c,0)+self.NEG.get(c,0)
        return max(0,min(1,s))
    def shap(self,t):
        sv={}
        for w in t.split():
            c=w.strip(".,!?;:").lower(); v=self.POS.get(c,0)+self.NEG.get(c,0)
            if v!=0: sv[c]=round(v,4)
        return sv
    def lime(self,t):
        ws=t.split(); b=self.predict(t); imp={}
        for i,w in enumerate(ws):
            p=" ".join(x for j,x in enumerate(ws) if j!=i)
            c=w.strip(".,!?;:").lower(); d=round(b-self.predict(p),4)
            if d!=0: imp[c]=d
        return imp
    def attn(self,t):
        ws=t.split(); cl=[w.strip(".,!?;:").lower() for w in ws]; n=len(ws)
        a=np.ones((n,n))*0.1
        for i in range(n):
            a[i,i]=0.3
            for j in range(n):
                if cl[j] in self.POS or cl[j] in self.NEG: a[i,j]+=0.4
        a=a/a.sum(1,keepdims=True); r=a.sum(0); r=r/r.sum()
        return {cl[i]:round(float(r[i]),4) for i in range(n)}
    def explain(self,t):
        sv=self.shap(t); lm=self.lime(t); at=self.attn(t)
        ts=max(sv,key=lambda k:abs(sv[k])) if sv else None
        tl=max(lm,key=lambda k:abs(lm[k])) if lm else None
        ta=max(at,key=lambda k:at[k]) if at else None
        return {"pred":round(self.predict(t),3),"ts":ts,"tl":tl,"ta":ta,"agree":ts==tl==ta}

d=XAI()
texts=["The course was excellent and the projects were great",
    "Terrible experience boring and confusing",
    "I love how clear and helpful the examples are",
    "Bad course disappointing waste of time",
    "This is the best amazing GenAI course fantastic"]

print("XAI Dashboard:")
ag=0
for i,t in enumerate(texts):
    r=d.explain(t); lb="POS" if r["pred"]>=0.5 else "NEG"
    tag = "Y" if r["agree"] else "N"
    print(f"  {i+1}. pred={r['pred']} ({lb}) SHAP={r['ts']} LIME={r['tl']} Attn={r['ta']} Agree={tag}")
    if r["agree"]: ag+=1
pct = ag/len(texts)*100
print(f"\n  Agreement: {ag}/{len(texts)} ({pct:.0f}%)")


</details>
